# 📘 Phần 2: Xử Lý Dữ Liệu Khuyết Thiếu (Handling Missing / Null Values in Pandas)

Dữ liệu khuyết thiếu (Missing Data) xảy ra khi thông tin không được lưu trữ hoặc bị lỗi trong quá trình thu thập. Nếu không xử lý đúng cách, các thuật toán thống kê và Machine Learning sẽ báo lỗi hoặc đưa ra kết quả thiếu chính xác.

---

## 🎯 Mục Tiêu Bài Học:
1. Phát hiện và thống kê missing value (`isna()`, `isnull()`, tỉ lệ %).
2. Chuẩn hóa các giá trị rỗng dạng chuỗi (`"N/A"`, `"?"`, `"-"`, `"null"`, `""`) về chuẩn `np.nan`.
3. Xóa dữ liệu khuyết thiếu an toàn với `dropna()` (`how`, `subset`, `thresh`).
4. Điền dữ liệu khuyết thiếu (Imputation):
   - Điền giá trị cố định (Constant).
   - Điền giá trị thống kê: **Mean**, **Median** (cho dữ liệu số có outlier), **Mode** (cho biến phân loại).
   - Điền chuỗi thời gian: **Forward fill (`ffill`)**, **Backward fill (`bfill`)**, **Nội suy (`interpolate`)**.
5. Kỹ thuật tạo cờ báo khuyết thiếu (Missingness Indicator Flag).


In [ ]:
import pandas as pd
import numpy as np

# 1. Đọc dữ liệu mẫu
df = pd.read_csv("data/customer_orders_raw.csv")
# Loại bỏ duplicate trước để tập trung vào missing values
df = df.drop_duplicates(subset=['order_id'], keep='first').reset_index(drop=True)
df.head(10)


## 1. Chuẩn Hóa Các Ký Tự Rỗng Dạng Chuỗi Về `np.nan`
Trong thực tế, người dùng hoặc hệ thống thường xuất ra các chuỗi như `"N/A"`, `"?"`, `"null"`, `"None"`, `"-"`, hoặc khoảng trắng rỗng `""`. Pandas mặc định có thể coi chúng là chuỗi hợp lệ chứ không phải NaN.


In [ ]:
# Danh sách các giá trị đại diện cho rỗng / không xác định
missing_markers = ['N/A', 'NaN', 'null', 'None', '?', '-', '']

# Thay thế toàn bộ các ký tự này thành np.nan
df.replace(missing_markers, np.nan, inplace=True)
# Xử lý thêm các chuỗi chỉ toàn khoảng trắng
df = df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)
df.replace('', np.nan, inplace=True)

print("Đã chuẩn hóa tất cả ký tự rỗng về np.nan!")


## 2. Thống Kê Dữ Liệu Khuyết Thiếu Theo Cột
Kiểm tra số lượng và phần trăm missing values ở từng cột để đưa ra chiến lược xử lý phù hợp.


In [ ]:
missing_summary = pd.DataFrame({
    'Missing_Count': df.isna().sum(),
    'Missing_Pct (%)': (df.isna().sum() / len(df) * 100).round(2)
})
missing_summary.sort_values(by='Missing_Count', ascending=False)


## 3. Chiến Lược 1: Xóa Dữ Liệu Khuyết Thiếu (`dropna()`)
- `dropna(how='any')`: Xóa dòng nếu có ÍT NHẤT 1 cột bị NaN.
- `dropna(how='all')`: Chỉ xóa dòng nếu TẤT CẢ các cột đều là NaN.
- `dropna(subset=['col1', 'col2'])`: Chỉ kiểm tra NaN trên các cột quan trọng.
- `dropna(thresh=N)`: Giữ lại dòng có ít nhất `N` giá trị không rỗng (non-null).


In [ ]:
# Ví dụ 1: Xóa các dòng bị thiếu thông tin liên hệ bắt buộc (email)
df_valid_email = df.dropna(subset=['email']).copy()
print(f"Số dòng trước khi drop email null: {len(df)}")
print(f"Số dòng sau khi drop email null:  {len(df_valid_email)}")

# Ví dụ 2: Giữ lại các dòng có ít nhất 10/12 cột có giá trị
df_thresh = df.dropna(thresh=10)
print(f"Số dòng thỏa mãn thresh=10: {len(df_thresh)}")


## 4. Chiến Lược 2: Điền Giá Trị (Imputation)

### A. Điền Giá Trị Cố Định (Constant)
Thường áp dụng cho thông tin tùy chọn hoặc phân loại không xác định (ví dụ: `phone` -> `'Unknown'`, `discount_rate` -> `'0%'`).


In [ ]:
df_imputed = df.copy()

# Điền cột phone bị thiếu bằng 'Unknown'
df_imputed['phone'] = df_imputed['phone'].fillna('Unknown')

# Điền cột discount_rate bị thiếu bằng '0%'
df_imputed['discount_rate'] = df_imputed['discount_rate'].fillna('0%')

print("Điền giá trị cố định thành công:")
df_imputed[['order_id', 'customer_name', 'phone', 'discount_rate']].head(8)


### B. Điền Giá Trị Thống Kê (Mean / Median / Mode)
- **Mean (Trung bình)**: Thích hợp cho dữ liệu số phân phối chuẩn, không có giá trị ngoại lai lớn.
- **Median (Trung vị)**: Thích hợp cho dữ liệu số có outlier (ví dụ tuổi, thu nhập, giá cả).
- **Mode (Yếu vị)**: Thích hợp cho dữ liệu danh mục / phân loại (ví dụ: thành phố, danh mục sản phẩm).


In [ ]:
# Chuyển đổi cột age sang số để tính toán (loại bỏ các giá trị không hợp lệ tạm thời)
df_imputed['age_num'] = pd.to_numeric(df_imputed['age'], errors='coerce')

# Tính median tuổi của những người có tuổi hợp lệ (10 - 100 tuổi)
valid_age_median = df_imputed.loc[(df_imputed['age_num'] >= 10) & (df_imputed['age_num'] <= 100), 'age_num'].median()
print(f"Median tuổi hợp lệ: {valid_age_median}")

# Điền median vào các ô bị missing
df_imputed['age_filled'] = df_imputed['age_num'].fillna(valid_age_median)

# Điền Mode cho cột city (thành phố xuất hiện nhiều nhất)
city_mode = df_imputed['city'].mode()[0]
print(f"Mode của cột city: {city_mode}")
df_imputed['city_filled'] = df_imputed['city'].fillna(city_mode)

df_imputed[['customer_name', 'age', 'age_filled', 'city', 'city_filled']].head(8)


### C. Điền Chuỗi Thời Gian: Forward Fill (`ffill`), Backward Fill (`bfill`), Interpolation
Thường dùng khi dữ liệu có tính tuần tự theo thời gian (giá chứng khoán, cảm biến nhiệt độ).


In [ ]:
# Tạo dữ liệu mẫu chuỗi thời gian
ts_demo = pd.Series([10.0, np.nan, np.nan, 25.0, np.nan, 40.0])

demo_df = pd.DataFrame({
    'Original': ts_demo,
    'ffill (Lấy giá trị liền trước)': ts_demo.ffill(),
    'bfill (Lấy giá trị liền sau)': ts_demo.bfill(),
    'Interpolate (Nội suy tuyến tính)': ts_demo.interpolate(method='linear')
})
demo_df


## 5. Kỹ Thuật Tạo Cờ Báo Khuyết Thiếu (Missingness Indicator)
Khi xây dựng mô hình Machine Learning, việc một trường bị thiếu đôi khi mang ý nghĩa thông tin quan trọng. Do đó, việc lưu lại cờ `is_missing` trước khi impute là một Best Practice.


In [ ]:
# Tạo cờ đánh dấu khách hàng có bị thiếu thông tin tuổi ban đầu hay không
df_imputed['is_age_missing'] = df['age'].isna().astype(int)

df_imputed[['order_id', 'customer_name', 'age', 'age_filled', 'is_age_missing']].head(10)


## 6. Tổng Kết Quy Trình Xử Lý Missing Values
1. Tìm hiểu nguyên nhân gây ra missing value (do lỗi hệ thống, người dùng bỏ qua hay không áp dụng).
2. Chuẩn hóa tất cả các chuỗi rỗng bất thường về `np.nan`.
3. Đánh giá % missing: Nếu cột bị missing > 70% và không quan trọng, cân nhắc xóa cột; nếu < 5%, có thể xóa dòng hoặc điền giá trị.
4. Chọn phương pháp Imputation phù hợp (Median cho số có outlier, Mode cho phân loại).
5. Tạo cờ chỉ báo missing nếu chuẩn bị cho Machine Learning.
